## Agent Skills with Claude Managed Agents

### Installing Utilities and Libraries

In [ ]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key = claude_api_key)

### Upload the Marketing Skill

In [ ]:
from anthropic.lib import files_from_dir

skill = client.beta.skills.create(
    files = files_from_dir("./skills/marketing-review")
)

print("Skill ID:", skill.id)
print("Latest Version:", skill.latest_version)

### Creating the Agent

In [ ]:
agent = client.beta.agents.create(
    name="Marketing-Agent",
    model=claude_model_name,
    system="You are a helpful AI Marketing Assistant.",
    skills = [
        {
            "type": "anthropic",
            "skill_id": "docx" 
        },
        {
            "type": "custom",
            "skill_id": skill.id,
            "version": skill.latest_version
        }
    ],
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

### Create an Environment

In [ ]:
environment = client.beta.environments.create(
    name="quickstart-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

### Start a Session

In [ ]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="Quickstart session",
)

print(f"Session ID: {session.id}")

### Execute the Agent

In [ ]:
with client.beta.sessions.events.stream(session.id) as stream:
            # Send the user message after the stream opens
            client.beta.sessions.events.send(
                session.id,
                betas = ["skills-2025-10-02"],
                events=[
                    {
                        "type": "user.message",
                        "content": [
                            {
                                "type": "text",
                                "text": """Create a professional LinkedIn product launch announcement for our new AI-powered fitness smartwatch, FitSense AI.

Use the company's branding guidelines, marketing checklist, and LinkedIn launch template to produce polished marketing content.

Generate the final deliverable as a professionally formatted Microsoft Word (.docx) document and save it as an output file.""",
                            },
                        ],
                    },
                ],
            )

            # Process streaming events
            for event in stream:
                match event.type:
                    case "agent.message":
                        for block in event.content:
                            print(block.text, end="")
                    case "agent.tool_use":
                        print(f"\n[Using tool: {event.name}]")
                    case "session.status_idle":
                        print("\n\nAgent finished.")
                        break

In [ ]:
files = client.beta.files.list(
    scope_id=session.id,
    betas=["managed-agents-2026-04-01"],
)

for file in files.data:

    if file.filename.endswith(".docx"):

        print(f"Downloading {file.filename}...")

        metadata = client.beta.files.retrieve_metadata(file.id)

        content = client.beta.files.download(file.id)

        content.write_to_file(metadata.filename)

        print("Done!")